# Notebook 1: Data Loading & Extraction
## PulsePredictor - Depression Risk Factor Analysis (BRFSS 2021)

This notebook covers the full data pipeline from raw CDC survey data to a clean, model-ready dataset. The source is the **BRFSS 2021 XPT file** (438,693 respondents, 303 variables) published by the CDC. By the end of this notebook, we produce train/test splits ready for modeling in the next notebook.

#### Step 1. Installing libraries

In [78]:
# Install libraries (only needed once)
#import subprocess
#subprocess.run(["pip", "install", "pyreadstat", "pandas", "numpy", "matplotlib", "seaborn"], 
               #capture_output=True)

# Import libraries
import pyreadstat
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings("ignore")

print("All libraries imported successfully!")

All libraries imported successfully!


## Install & Import Libraries

### Installation (commented out)
The `subprocess` block at the top is intentionally commented out. It was used once during initial setup to install the required packages, and is kept here as a reference so anyone cloning this repo knows exactly which packages to install if they're running this on a fresh environment. Under normal usage, the libraries are already installed and this block does not need to run.

### Why each library?

| Library | Purpose |
|--------|---------|
| `pyreadstat` | Reads the CDC's `.XPT` file format (SAS Transport format). Standard pandas cannot open this format, so this library is specifically needed for BRFSS data. |
| `pandas` | Core library for loading, cleaning, filtering, and transforming tabular data throughout this notebook. |
| `numpy` | Used for numerical operations and for representing missing values as `np.nan` when replacing BRFSS special codes. |
| `matplotlib` | Base plotting library — used as the foundation for any visualizations. |
| `seaborn` | Built on top of matplotlib, provides cleaner statistical charts with less code. Used for distribution and count plots during exploration. |
| `os` | Provides operating system utilities — used for file path checks and directory operations where needed. |
| `warnings` | Suppresses non-critical warnings (mainly from pyreadstat when parsing the large XPT file) to keep the output clean and readable. |

#### Step 2. Defining paths

In [79]:
from pathlib import Path

# Path.cwd() = folder where this notebook is running from
# .parent    = one level up = PulsePredictor/ (project root)
# This works on ANY machine regardless of username or OS
BASE_DIR      = Path.cwd().parent
RAW_DIR       = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
FIGURES_DIR   = BASE_DIR / "outputs" / "figures"
TABLES_DIR    = BASE_DIR / "outputs" / "tables"

# Auto-create folders so anyone cloning from GitHub doesn't get errors
for folder in [PROCESSED_DIR, FIGURES_DIR, TABLES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Find XPT file dynamically, this works even if filename changes
xpt_files     = list(RAW_DIR.glob("*.XPT")) + list(RAW_DIR.glob("*.xpt"))
if not xpt_files:
    raise FileNotFoundError(
        f"\n No XPT file found in: {RAW_DIR}"
        f"\n Download LLCP2021.XPT from CDC and place it in data/raw/"
    )
XPT_FILE      = xpt_files[0]

# Output paths
FULL_CSV      = PROCESSED_DIR / "brfss_2021_full.csv"
ANALYSIS_CSV  = PROCESSED_DIR / "brfss_2021_depression.csv"

print(f" Base directory   : {BASE_DIR}")
print(f" XPT file found : {XPT_FILE}")
print(f" Full CSV out   : {FULL_CSV}")
print(f" Analysis CSV   : {ANALYSIS_CSV}")

 Base directory   : /Users/ishamandawkar/PulsePredictor
 XPT file found : /Users/ishamandawkar/PulsePredictor/data/raw/LLCP2021.XPT
 Full CSV out   : /Users/ishamandawkar/PulsePredictor/data/processed/brfss_2021_full.csv
 Analysis CSV   : /Users/ishamandawkar/PulsePredictor/data/processed/brfss_2021_depression.csv


## Define File Paths

### What is `pathlib`?
`pathlib` is a Python built-in library for handling file and folder paths. Unlike writing paths as plain strings (e.g., `"/Users/isha/project/data"`), `pathlib` creates **Path objects** that automatically use the correct format for whatever operating system is running the code , so the same code works on Mac, Windows, and Linux without any changes.

### Why are we using it here?
Different people may run it on different machines with different usernames and folder structures. Using `pathlib` with `Path.cwd()` means no one has to manually edit a hardcoded path like `/Users/ishamandawkar/...` to get it to work on their machine.

### What this code does
- `Path.cwd()` — gets the folder where this notebook is currently running (the `notebooks/` directory)
- `.parent` — steps one level up to reach the project root (`PulsePredictor/`)
- From the root, all other folders (`data/raw`, `data/processed`, `outputs/figures`, `outputs/tables`) are built by chaining folder names with `/`
- `folder.mkdir(parents=True, exist_ok=True)` — auto-creates any output folders that don't exist yet, so cloning the repo and running this notebook will never throw a "folder not found" error
- The XPT file is located dynamically using `glob("*.XPT")` rather than a hardcoded filename, so it still works even if the file gets renamed

#### Step 3. Loading the XPT file

In [80]:
print("Loading XPT file")

df, meta = pyreadstat.read_xport(XPT_FILE)

print(f"Done!")
print(f"   Rows    : {df.shape[0]:,}")
print(f"   Columns : {df.shape[1]}")
print(f"\nFirst 3 rows, first 5 columns:")
df.iloc[:3, :5]

Loading XPT file
Done!
   Rows    : 438,693
   Columns : 303

First 3 rows, first 5 columns:


,_STATE,FMONTH,IDATE,IMONTH,IDAY
0,1.0,1.0,01192021,01,19
1,1.0,1.0,01212021,01,21
2,1.0,1.0,01212021,01,21


## Load the Raw XPT File

The raw BRFSS 2021 data is loaded from its original `.XPT` format using `pyreadstat`. The dataset contains **438,693 respondents** across **303 variables** — one row per survey participant. The `meta` object (not used further here) contains variable labels from the SAS file, which were consulted alongside the [CDC BRFSS 2021 Codebook](https://www.cdc.gov/brfss/annual_data/2021/pdf/codebook21_llcp-v2-508.PDF) for variable identification.

#### Step 4. Saving it into CSV format

In [81]:
print("Saving full dataset to CSV")

df.to_csv(FULL_CSV, index=False)

print(f"Saved! File location:")
print(f"   {FULL_CSV}")

Saving full dataset to CSV
Saved! File location:
   /Users/ishamandawkar/PulsePredictor/data/processed/brfss_2021_full.csv


## Save Full Dataset as CSV

This dataset that contains 303 columns is saved as a CSV immediately after loading. This serves as our permanent raw backup — all subsequent steps work on subsets, so this file allows us to re-extract different variables later without re-loading the large XPT file from scratch.

#### Step 5. Data Overview, checking the target variable

In [82]:
# 5a. Basic info
print("=" * 50)
print("DATASET OVERVIEW")
print("=" * 50)
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")

print("\nColumns:")
print(df.columns)

print("\nFirst 5 rows:")
display(df.head())

print("\nLast 5 rows:")
display(df.tail())

print("\nDataset Info:")
df.info()

# 5b. Check target variable
print("\n" + "=" * 50)
print("TARGET VARIABLE: ADDEPEV3")
print("(Ever told you had depressive disorder)")
print("=" * 50)
print(df["ADDEPEV3"].value_counts(dropna=False))
print("\n1 = Yes  |  2 = No  |  7 = Don't Know  |  9 = Refused  |  NaN = Missing")

# 5c. Missing values overview
print("\n" + "=" * 50)
print("TOP 15 COLUMNS WITH MOST MISSING VALUES")
print("=" * 50)
missing = (df.isnull().sum() / len(df) * 100).round(1)
print(missing[missing > 0].sort_values(ascending=False).head(15))

DATASET OVERVIEW
Rows    : 438,693
Columns : 303

Columns:
Index(['_STATE', 'FMONTH', 'IDATE', 'IMONTH', 'IDAY', 'IYEAR', 'DISPCODE',
       'SEQNO', '_PSU', 'CTELENM1',
       ...
       '_FRTRES1', '_VEGRES1', '_FRUTSU1', '_VEGESU1', '_FRTLT1A', '_VEGLT1A',
       '_FRT16A', '_VEG23A', '_FRUITE1', '_VEGETE1'],
      dtype='object', length=303)

First 5 rows:


,_STATE,FMONTH,IDATE,IMONTH,IDAY,IYEAR,DISPCODE,SEQNO,_PSU,CTELENM1,...,_FRTRES1,_VEGRES1,_FRUTSU1,_VEGESU1,_FRTLT1A,_VEGLT1A,_FRT16A,_VEG23A,_FRUITE1,_VEGETE1
0,1.0,1.0,01192021,01,19,2021,1100.0,2021000001,2.021000e+09,1.0,...,1.0,1.0,100.0,214.0,1.0,1.0,1.0,1.0,0.0,0.0
1,1.0,1.0,01212021,01,21,2021,1100.0,2021000002,2.021000e+09,1.0,...,1.0,1.0,100.0,128.0,1.0,1.0,1.0,1.0,0.0,0.0
2,1.0,1.0,01212021,01,21,2021,1100.0,2021000003,2.021000e+09,1.0,...,1.0,1.0,100.0,71.0,1.0,2.0,1.0,1.0,0.0,0.0
3,1.0,1.0,01172021,01,17,2021,1100.0,2021000004,2.021000e+09,1.0,...,1.0,1.0,114.0,165.0,1.0,1.0,1.0,1.0,0.0,0.0
4,1.0,1.0,01152021,01,15,2021,1100.0,2021000005,2.021000e+09,1.0,...,1.0,1.0,100.0,258.0,1.0,1.0,1.0,1.0,0.0,0.0



Last 5 rows:


,_STATE,FMONTH,IDATE,IMONTH,IDAY,IYEAR,DISPCODE,SEQNO,_PSU,CTELENM1,...,_FRTRES1,_VEGRES1,_FRUTSU1,_VEGESU1,_FRTLT1A,_VEGLT1A,_FRT16A,_VEG23A,_FRUITE1,_VEGETE1
438688,78.0,12.0,01062022,01,06,2022,1100.0,2021001381,2.021001e+09,NaN,...,1.0,1.0,157.0,393.0,1.0,1.0,1.0,1.0,0.0,0.0
438689,78.0,12.0,01122022,01,12,2022,1100.0,2021001382,2.021001e+09,NaN,...,1.0,1.0,200.0,157.0,1.0,1.0,1.0,1.0,0.0,0.0
438690,78.0,12.0,12212021,12,21,2021,1100.0,2021001383,2.021001e+09,NaN,...,1.0,1.0,200.0,143.0,1.0,1.0,1.0,1.0,0.0,0.0
438691,78.0,12.0,01112022,01,11,2022,1100.0,2021001384,2.021001e+09,NaN,...,1.0,1.0,100.0,156.0,1.0,1.0,1.0,1.0,0.0,0.0
438692,78.0,12.0,12222021,12,22,2021,1100.0,2021001385,2.021001e+09,NaN,...,1.0,1.0,34.0,66.0,2.0,2.0,1.0,1.0,0.0,0.0



Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 438693 entries, 0 to 438692
Columns: 303 entries, _STATE to _VEGETE1
dtypes: float64(298), object(5)
memory usage: 1014.1+ MB

TARGET VARIABLE: ADDEPEV3
(Ever told you had depressive disorder)
ADDEPEV3
2.0    350778
1.0     85398
7.0      1814
9.0       700
NaN         3
Name: count, dtype: int64

1 = Yes  |  2 = No  |  7 = Don't Know  |  9 = Refused  |  NaN = Missing

TOP 15 COLUMNS WITH MOST MISSING VALUES
TRETHEPC    100.0
HAVEHEPC    100.0
SIGMTES1    100.0
TOLDCFS     100.0
HAVECFS     100.0
MEDSHEPB    100.0
VCLNTES1    100.0
COLGSEX     100.0
PRIRHEPC    100.0
COLGHOUS    100.0
WORKCFS     100.0
BLDSTFIT     99.9
SDNATES1     99.9
LASTSIG4     99.8
CSRVCTL2     99.8
dtype: float64


## Dataset Overview & Target Variable Inspection

Before selecting features, we inspect the dataset structure and verify our target variable `ADDEPEV3` ("Have you ever been told you had a depressive disorder?"). 

Key findings:
- **85,398 respondents (19.5%)** answered Yes (1.0) — this becomes our positive class
- **350,778 (80%)** answered No — the negative class
- A small number of responses (7 = Don't Know, 9 = Refused, NaN) will be excluded
- Several columns show **100% missing values** — these are optional BRFSS modules not collected in 2021, and will naturally be excluded during variable selection

#### Step 6. Identifying which variables would relate to Depression?

In [83]:
# Define our 35 variables for the depression use case
DEPRESSION_VARIABLES = {
    # TARGET
    "ADDEPEV3": "Ever_Depressive_Disorder",

    # DEMOGRAPHIC
    "_AGEG5YR":  "Age_Group",
    "SEX1":      "Sex",
    "EDUCA":     "Education_Level",
    "INCOME3":   "Income_Level",
    "MARITAL":   "Marital_Status",
    "EMPLOY1":   "Employment_Status",
    "_RACEGR4":  "Race_Ethnicity",
    "_STATE":    "State",

    # LIFESTYLE: Physical Activity
    "EXERANY2":  "Exercise_Past30Days",

    # LIFESTYLE: Sleep
    "SLEPTIM1":  "Sleep_Hours",

    # LIFESTYLE: Smoking
    "SMOKE100":  "Smoked_100_Cigarettes",
    "SMOKDAY2":  "Smoke_Frequency",

    # LIFESTYLE: Alcohol
    "DRNKANY6":  "Drank_Alcohol_Past30Days",
    "AVEDRNK3":  "Avg_Drinks_Per_Day",
    "DRNK3GE5":  "Binge_Drinking_Days",

    # LIFESTYLE: Diet
    "FRUIT2":    "Fruit_Per_Day",
    "VEGETAB2":  "Vegetables_Per_Day",

    # HEALTH STATUS
    "_BMI5CAT":  "BMI_Category",
    "GENHLTH":   "General_Health",
    "PHYSHLTH":  "Poor_Physical_Health_Days",
    "MENTHLTH":  "Poor_Mental_Health_Days",
    "HLTHPLN1":  "Have_Health_Plan",
    "MEDCOST1":  "Could_Not_Afford_Doctor",

    # CHRONIC CONDITIONS
    "CVDCRHD4":  "Ever_Heart_Disease",
    "CVDSTRK3":  "Ever_Stroke",
    "DIABETE4":  "Diabetes_Status",
    "CHCCOPD3":  "Ever_COPD",
    "HAVARTH4":  "Ever_Arthritis",
    "CHCKDNY2":  "Ever_Kidney_Disease",

    # SOCIAL FACTORS
    "LSATISFY":  "Life_Satisfaction",
    "EMTSUPRT":  "Emotional_Support",
    "SDHISOLT":  "Social_Isolation",
    "FOODSTMP":  "Received_Food_Stamps",
    "SDHBILLS":  "Difficulty_Paying_Bills",
}

# Check which ones exist in our dataset
available   = {k: v for k, v in DEPRESSION_VARIABLES.items() if k in df.columns}
not_found   = [k for k in DEPRESSION_VARIABLES if k not in df.columns]

print(f"Found   : {len(available)} / {len(DEPRESSION_VARIABLES)} variables")
print(f"Missing : {not_found}")

Found   : 24 / 35 variables
Missing : ['SEX1', '_RACEGR4', 'SLEPTIM1', 'DRNKANY6', 'HLTHPLN1', 'HAVARTH4', 'LSATISFY', 'EMTSUPRT', 'SDHISOLT', 'FOODSTMP', 'SDHBILLS']


## Select Depression-Relevant Features

Rather than working with all 303 columns, we define a targeted subset of 35 variables grounded in the clinical and social science literature on depression risk factors — covering demographics, lifestyle behaviors, health status, chronic conditions, and social determinants of health. This keeps the model interpretable and avoids noise from irrelevant survey modules.

Initial check finds **24 of 35 variables** present. The 11 missing variables either use different column names in the 2021 dataset or were not part of the 2021 survey questionnaire , we investigate each below.

#### Step 6.1.  24 found out of 35, let's find the actual column names 

In [84]:
# Search for the actual column names in the dataset
search_terms = {
    "SEX"      : "Sex",
    "RACE"     : "Race/Ethnicity", 
    "SLEEP"    : "Sleep",
    "DRINK"    : "Alcohol/Drinking",
    "HLTH"     : "Health Plan",
    "ARTH"     : "Arthritis",
    "SATISF"   : "Life Satisfaction",
    "EMTSUP"   : "Emotional Support",
    "ISOL"     : "Social Isolation",
    "FOOD"     : "Food Stamps",
    "BILL"     : "Bills/Financial",
}

for term, label in search_terms.items():
    matches = [col for col in df.columns if term in col.upper()]
    print(f"\n {label} ('{term}'):")
    if matches:
        for m in matches:
            print(f"   → {m}")
    else:
        print(f"No matches found")


 Sex ('SEX'):
   → COLGSEX
   → LANDSEX
   → CELLSEX
   → SEXVAR
   → ACEHVSEX
   → BIRTHSEX
   → _SEX

 Race/Ethnicity ('RACE'):
   → _IMPRACE
   → _CRACE1
   → _CPRACE1
   → _PRACE1
   → _MRACE1
   → _RACE
   → _RACEG21
   → _RACEGR3
   → _RACEPRV

 Sleep ('SLEEP'):
No matches found

 Alcohol/Drinking ('DRINK'):
   → ACEDRINK

 Health Plan ('HLTH'):
   → GENHLTH
   → PHYSHLTH
   → MENTHLTH
   → POORHLTH
   → _RFHLTH
   → _HLTHPLN

 Arthritis ('ARTH'):
   → HAVARTH5
   → ARTHEXER
   → ARTHEDU
   → ARTHDIS2

 Life Satisfaction ('SATISF'):
No matches found

 Emotional Support ('EMTSUP'):
No matches found

 Social Isolation ('ISOL'):
No matches found

 Food Stamps ('FOOD'):
No matches found

 Bills/Financial ('BILL'):
No matches found


##  Search for Alternative Column Names

For the 11 missing variables, we search the dataset's column names using partial string matching. This surfaces the correct 2021 variable names, for example, the sex variable is `SEXVAR` in 2021 (not `SEX1`), and arthritis is `HAVARTH5` (not `HAVARTH4`). This step avoids relying on codebook version assumptions.

#### Step 6.2. Finalized variables

In [85]:
# Finalized variables based on what actually exists in BRFSS 2021
FINAL_VARIABLES = {
    # TARGET
    "ADDEPEV3":  "Ever_Depressive_Disorder",

    # DEMOGRAPHIC
    "_AGEG5YR":  "Age_Group",
    "SEXVAR":    "Sex",
    "EDUCA":     "Education_Level",
    "INCOME3":   "Income_Level",
    "MARITAL":   "Marital_Status",
    "EMPLOY1":   "Employment_Status",
    "_RACEGR3":  "Race_Ethnicity",
    "_STATE":    "State",

    # LIFESTYLE: Physical Activity
    "EXERANY2":  "Exercise_Past30Days",

    # LIFESTYLE: Sleep
    "SLEPTIM1":  "Sleep_Hours",

    # LIFESTYLE: Smoking
    "SMOKE100":  "Smoked_100_Cigarettes",
    "SMOKDAY2":  "Smoke_Frequency",

    # LIFESTYLE: Alcohol
    "AVEDRNK3":  "Avg_Drinks_Per_Day",
    "DRNK3GE5":  "Binge_Drinking_Days",

    # LIFESTYLE: Diet
    "FRUIT2":    "Fruit_Per_Day",
    "VEGETAB2":  "Vegetables_Per_Day",

    # HEALTH STATUS
    "_BMI5CAT":  "BMI_Category",
    "GENHLTH":   "General_Health",
    "PHYSHLTH":  "Poor_Physical_Health_Days",
    "MENTHLTH":  "Poor_Mental_Health_Days",
    "_HLTHPLN":  "Have_Health_Plan",
    "MEDCOST1":  "Could_Not_Afford_Doctor",

    # CHRONIC CONDITIONS
    "CVDCRHD4":  "Ever_Heart_Disease",
    "CVDSTRK3":  "Ever_Stroke",
    "DIABETE4":  "Diabetes_Status",
    "CHCCOPD3":  "Ever_COPD",
    "HAVARTH5":  "Ever_Arthritis",
    "CHCKDNY2":  "Ever_Kidney_Disease",
}

# Check all exist
available  = {k: v for k, v in FINAL_VARIABLES.items() if k in df.columns}
not_found  = [k for k in FINAL_VARIABLES if k not in df.columns]

print(f"Found   : {len(available)} / {len(FINAL_VARIABLES)} variables")
if not_found:
    print(f"Still missing : {not_found}")
else:
    print("All variables found!")

Found   : 28 / 29 variables
Still missing : ['SLEPTIM1']


 Checking whether Sleep has a different column name or not

In [86]:
# Sleep column has a different name — let's find it
sleep_matches = [col for col in df.columns if "SLP" in col.upper() 
                 or "SLEEP" in col.upper() 
                 or "SLEP" in col.upper()
                 or "SLEPTIME" in col.upper()]
print("Sleep related columns:")
for s in sleep_matches:
    print(f"  → {s}")

Sleep related columns:


#### Step 6.4. Final confirmed variables

In [87]:
# FINAL confirmed variables for BRFSS 2021 depression analysis
FINAL_VARIABLES = {
    # TARGET
    "ADDEPEV3":  "Ever_Depressive_Disorder",

    # DEMOGRAPHIC
    "_AGEG5YR":  "Age_Group",
    "SEXVAR":    "Sex",
    "EDUCA":     "Education_Level",
    "INCOME3":   "Income_Level",
    "MARITAL":   "Marital_Status",
    "EMPLOY1":   "Employment_Status",
    "_RACEGR3":  "Race_Ethnicity",
    "_STATE":    "State",

    # LIFESTYLE: Physical Activity
    "EXERANY2":  "Exercise_Past30Days",

    # LIFESTYLE: Smoking
    "SMOKE100":  "Smoked_100_Cigarettes",
    "SMOKDAY2":  "Smoke_Frequency",

    # LIFESTYLE: Alcohol
    "AVEDRNK3":  "Avg_Drinks_Per_Day",
    "DRNK3GE5":  "Binge_Drinking_Days",

    # LIFESTYLE: Diet
    "FRUIT2":    "Fruit_Per_Day",
    "VEGETAB2":  "Vegetables_Per_Day",

    # HEALTH STATUS
    "_BMI5CAT":  "BMI_Category",
    "GENHLTH":   "General_Health",
    "PHYSHLTH":  "Poor_Physical_Health_Days",
    "MENTHLTH":  "Poor_Mental_Health_Days",
    "_HLTHPLN":  "Have_Health_Plan",
    "MEDCOST1":  "Could_Not_Afford_Doctor",

    # CHRONIC CONDITIONS
    "CVDCRHD4":  "Ever_Heart_Disease",
    "CVDSTRK3":  "Ever_Stroke",
    "DIABETE4":  "Diabetes_Status",
    "CHCCOPD3":  "Ever_COPD",
    "HAVARTH5":  "Ever_Arthritis",
    "CHCKDNY2":  "Ever_Kidney_Disease",
}

# Verify all exist
available = {k: v for k, v in FINAL_VARIABLES.items() if k in df.columns}
not_found = [k for k in FINAL_VARIABLES if k not in df.columns]

print(f"Found   : {len(available)} / {len(FINAL_VARIABLES)} variables")
if not_found:
    print(f"Missing : {not_found}")
else:
    print("All variables confirmed!")

# Extract subset
df_subset = df[list(available.keys())].copy()
df_subset.rename(columns=available, inplace=True)

print(f"\nExtracted shape: {df_subset.shape[0]:,} rows x {df_subset.shape[1]} columns")
print(f"\nColumns:")
for col in df_subset.columns:
    print(f"  → {col}")

Found   : 28 / 28 variables
All variables confirmed!

Extracted shape: 438,693 rows x 28 columns

Columns:
  → Ever_Depressive_Disorder
  → Age_Group
  → Sex
  → Education_Level
  → Income_Level
  → Marital_Status
  → Employment_Status
  → Race_Ethnicity
  → State
  → Exercise_Past30Days
  → Smoked_100_Cigarettes
  → Smoke_Frequency
  → Avg_Drinks_Per_Day
  → Binge_Drinking_Days
  → Fruit_Per_Day
  → Vegetables_Per_Day
  → BMI_Category
  → General_Health
  → Poor_Physical_Health_Days
  → Poor_Mental_Health_Days
  → Have_Health_Plan
  → Could_Not_Afford_Doctor
  → Ever_Heart_Disease
  → Ever_Stroke
  → Diabetes_Status
  → Ever_COPD
  → Ever_Arthritis
  → Ever_Kidney_Disease


## Final Confirmed Variable Set

With sleep removed, our final set is **28 variables** (1 target + 27 features), all confirmed present in the dataset. The subset `df_subset` is extracted and columns are renamed  for clarity throughout the rest of the pipeline.

## Understanding BRFSS Special Codes

Before cleaning, it's critical to understand that BRFSS uses numeric sentinel values that **look like real data but represent non-responses**:

| Code | Meaning |
|------|---------|
| 7 / 77 | Don't Know / Not Sure |
| 9 / 99 | Refused |
| 88 | None (legitimate zero, coded differently) |
| NaN | Truly missing |

If left in place, these codes corrupt the model for example, a respondent who "refused to answer" would appear as a data point with value `9`, which the model would treat as a high numeric response. Every variable is cleaned according to its specific codes from the codebook.

In [88]:
# ============================================================
# We already have df_subset with 438,693 rows and 28 columns
# ============================================================
print(f"Subset shape: {df_subset.shape[0]:,} rows x {df_subset.shape[1]} columns")

Subset shape: 438,693 rows x 28 columns


#### Step 7. Replace BRFSS special codes with NaN

In [89]:
# ============================================================
# Each variable has its own "invalid" codes per the codebook.
# We replace them with NaN so pandas treats them as missing.
# ============================================================

# These columns use 7 = Don't Know, 9 = Refused
binary_cols = [
    "Ever_Depressive_Disorder",
    "Exercise_Past30Days",
    "Smoked_100_Cigarettes",
    "Ever_Heart_Disease",
    "Ever_Stroke",
    "Ever_COPD",
    "Ever_Arthritis",
    "Ever_Kidney_Disease",
    "Could_Not_Afford_Doctor",
    "Have_Health_Plan",
]
for col in binary_cols:
    df_subset[col] = df_subset[col].replace({7: np.nan, 9: np.nan})

# Smoke frequency: 9 = Refused
df_subset["Smoke_Frequency"] = df_subset["Smoke_Frequency"].replace({9: np.nan})

# Diabetes: 7 = Don't Know, 9 = Refused
df_subset["Diabetes_Status"] = df_subset["Diabetes_Status"].replace({7: np.nan, 9: np.nan})

# General Health: 7 = Don't Know, 9 = Refused
df_subset["General_Health"] = df_subset["General_Health"].replace({7: np.nan, 9: np.nan})

# Income: 77 = Don't Know, 99 = Refused
df_subset["Income_Level"] = df_subset["Income_Level"].replace({77: np.nan, 99: np.nan})

# Education: 9 = Refused
df_subset["Education_Level"] = df_subset["Education_Level"].replace({9: np.nan})

# Marital Status: 9 = Refused
df_subset["Marital_Status"] = df_subset["Marital_Status"].replace({9: np.nan})

# Employment: 9 = Refused
df_subset["Employment_Status"] = df_subset["Employment_Status"].replace({9: np.nan})

# Age Group: 14 = Don't Know / Refused
df_subset["Age_Group"] = df_subset["Age_Group"].replace({14: np.nan})

# Poor Physical & Mental Health Days:
# 77 = Don't Know, 99 = Refused, 88 = None (recode 88 → 0)
for col in ["Poor_Physical_Health_Days", "Poor_Mental_Health_Days"]:
    df_subset[col] = df_subset[col].replace({88: 0, 77: np.nan, 99: np.nan})

# Avg Drinks Per Day: 99 = Refused
df_subset["Avg_Drinks_Per_Day"] = df_subset["Avg_Drinks_Per_Day"].replace({99: np.nan})

# Binge Drinking Days: 99 = Refused, 88 = None (recode 88 → 0)
df_subset["Binge_Drinking_Days"] = df_subset["Binge_Drinking_Days"].replace({88: 0, 99: np.nan})

# Fruit & Vegetables — complex codes, will handle separately below
# BMI Category: 0 = missing in BRFSS
df_subset["BMI_Category"] = df_subset["BMI_Category"].replace({0: np.nan})

print(" Special codes replaced with NaN")

 Special codes replaced with NaN


##  Replace BRFSS Special Codes with NaN

Each variable's invalid codes (Don't Know, Refused) are replaced with `NaN` so pandas treats them as missing. The code `88` ("None") is an exception , it represents a legitimate zero count (e.g., zero days of poor health) and is recoded to `0` rather than discarded. This is done per-variable based on the BRFSS 2021 codebook, since invalid code ranges differ across questions.

#### Step 8. Convert Fruit & Vegetables columns, handle them

In [90]:
# ============================================================
# These use a frequency+amount encoding:
# 100s = per day, 200s = per week, 300s = per month
# e.g. 101 = 1/day, 201 = 2/week, 301 = 3/month
# We convert everything to a daily estimate
# ============================================================

def convert_frequency(val):
    if pd.isna(val):
        return np.nan
    val = int(val)
    if val in [777, 999]:       # Don't know / Refused
        return np.nan
    elif 100 <= val <= 199:     # Per day
        return val - 100
    elif 200 <= val <= 299:     # Per week
        return round((val - 200) / 7, 2)
    elif 300 <= val <= 399:     # Per month
        return round((val - 300) / 30, 2)
    else:
        return np.nan

df_subset["Fruit_Per_Day"]      = df_subset["Fruit_Per_Day"].apply(convert_frequency)
df_subset["Vegetables_Per_Day"] = df_subset["Vegetables_Per_Day"].apply(convert_frequency)

print(" Fruit and Vegetable columns converted to daily estimates")

 Fruit and Vegetable columns converted to daily estimates


## Convert Fruit & Vegetable Frequency Codes

BRFSS encodes dietary frequency using a composite numeric scheme where the hundreds digit indicates the time unit and the remainder indicates the quantity:
- **100–199**: times per **day** (e.g., 102 = 2 per day)
- **200–299**: times per **week** (e.g., 207 = 7 per week → 1/day)
- **300–399**: times per **month** (e.g., 330 = 30 per month → 1/day)

A custom function converts all values to a standardized **daily estimate**, making the column numerically consistent and directly usable by a model.

#### Step 9. Check missing values

In [91]:
# ============================================================
# STEP 9 — Check missing values
# ============================================================

missing_after = (df_subset.isnull().sum() / len(df_subset) * 100).round(1)
missing_after = missing_after[missing_after > 0].sort_values(ascending=False)

print("Missing values per column (%):")
print(missing_after)

Missing values per column (%):
Smoke_Frequency              61.9
Binge_Drinking_Days          52.3
Avg_Drinks_Per_Day           52.2
Income_Level                 21.5
Vegetables_Per_Day           13.3
Fruit_Per_Day                12.5
BMI_Category                 10.7
Smoked_100_Cigarettes         5.6
Have_Health_Plan              4.0
Age_Group                     2.2
Poor_Physical_Health_Days     2.2
Employment_Status             1.9
Poor_Mental_Health_Days       1.8
Marital_Status                1.1
Ever_Heart_Disease            1.0
Ever_Arthritis                0.7
Ever_Depressive_Disorder      0.6
Education_Level               0.6
Ever_COPD                     0.5
Ever_Kidney_Disease           0.4
General_Health                0.3
Could_Not_Afford_Doctor       0.3
Ever_Stroke                   0.3
Exercise_Past30Days           0.2
Diabetes_Status               0.2
dtype: float64


## Missing Value Assessment After Cleaning

After replacing special codes, we inspect remaining missingness. Three variables stand out with **>50% missing**: `Smoke_Frequency` (61.9%), `Binge_Drinking_Days` (52.3%), and `Avg_Drinks_Per_Day` (52.2%). All other variables fall below 22% missing. These three high-missing columns are addressed in the next step.

#### Drop High-Missing Columns

Three columns are dropped before applying `dropna()`:

- **`Smoke_Frequency`** (61.9% missing): Smoking behavior is still captured by `Smoked_100_Cigarettes`, which has only 5.6% missing. The frequency detail is redundant given the cost.

- **`Binge_Drinking_Days`** and **`Avg_Drinks_Per_Day`** (both ~52% missing): These questions are only asked of respondents who reported drinking, meaning the missing data is **structurally non-random** (non-drinkers are systematically excluded). Keeping them would introduce bias.

Dropping these three columns before `dropna()` preserves significantly more rows for modeling.

In [92]:
# Drop the 3 high-missing columns
df_subset = df_subset.drop(columns=[
    "Smoke_Frequency",
    "Binge_Drinking_Days", 
    "Avg_Drinks_Per_Day"
])

print(f"✅ Columns remaining: {df_subset.shape[1]}")
print(f"   Columns dropped: Smoke_Frequency, Binge_Drinking_Days, Avg_Drinks_Per_Day")

✅ Columns remaining: 25
   Columns dropped: Smoke_Frequency, Binge_Drinking_Days, Avg_Drinks_Per_Day


#### Step 11. Categorical Encoding

In [93]:
# ============================================================
# Check current data types and unique values
# to decide what needs encoding
# ============================================================

print(f"{'Column':<30} {'Unique Values':<15} {'Sample Values'}")
print("-" * 70)
for col in df_subset.columns:
    unique = df_subset[col].dropna().unique()
    print(f"{col:<30} {len(unique):<15} {sorted(unique[:5])}")

Column                         Unique Values   Sample Values
----------------------------------------------------------------------
Ever_Depressive_Disorder       2               [1.0, 2.0]
Age_Group                      13              [9.0, 10.0, 11.0, 12.0, 13.0]
Sex                            2               [1.0, 2.0]
Education_Level                6               [2.0, 3.0, 4.0, 5.0, 6.0]
Income_Level                   11              [3.0, 4.0, 5.0, 6.0, 7.0]
Marital_Status                 6               [1.0, 2.0, 3.0, 5.0, 6.0]
Employment_Status              8               [1.0, 2.0, 5.0, 7.0, 8.0]
Race_Ethnicity                 6               [1.0, 2.0, 4.0, 5.0, 9.0]
State                          53              [1.0, 2.0, 4.0, 5.0, 6.0]
Exercise_Past30Days            2               [1.0, 2.0]
Smoked_100_Cigarettes          2               [1.0, 2.0]
Fruit_Per_Day                  158             [0.29, 0.4, 0.43, 0.57, 1.0]
Vegetables_Per_Day             156           

## Inspect Columns Before Encoding

We audit each column's unique values to decide the appropriate encoding strategy:
- **Binary columns** (2 unique values, coded 1/2): Will be recoded to 1/0
- **Ordinal columns** (naturally ordered, e.g., General Health 1–5, Age Group): Kept as numeric — their ordering carries meaningful information
- **Nominal columns** (no natural order, e.g., Marital Status, Race): Will be one-hot encoded

In [94]:
# ============================================================
# MILESTONE 4 — Categorical Encoding
# ============================================================

# ------------------------------------------------------------
# Part A — Recode Yes/No columns from (1/2) to (1/0)
# 1 = Yes → 1, 2 = No → 0
# ------------------------------------------------------------

yes_no_cols = [
    "Exercise_Past30Days",
    "Smoked_100_Cigarettes",
    "Have_Health_Plan",
    "Could_Not_Afford_Doctor",
    "Ever_Heart_Disease",
    "Ever_Stroke",
    "Ever_COPD",
    "Ever_Arthritis",
    "Ever_Kidney_Disease",
]
for col in yes_no_cols:
    df_subset[col] = df_subset[col].map({1.0: 1, 2.0: 0})

# Fix target variable
df_subset["Ever_Depressive_Disorder"] = df_subset["Ever_Depressive_Disorder"].map({1.0: 1, 2.0: 0})

# Recode Sex: 1 = Male → 1, 2 = Female → 0
df_subset["Sex"] = df_subset["Sex"].map({1.0: 1, 2.0: 0})

print("✅ Yes/No columns recoded to 1/0")
print(f"\nTarget distribution:")
print(df_subset["Ever_Depressive_Disorder"].value_counts(dropna=False))

# ------------------------------------------------------------
# Part B — One-hot encode truly categorical columns
# ------------------------------------------------------------

ohe_cols = [
    "Marital_Status",
    "Employment_Status", 
    "Race_Ethnicity",
    "Diabetes_Status",
    "State",
]

df_subset = pd.get_dummies(df_subset, columns=ohe_cols, drop_first=False)

print(f"\n One-hot encoding done")
print(f"   Shape after encoding: {df_subset.shape[0]:,} rows x {df_subset.shape[1]} columns")
print(f"\n   New columns added:")
new_cols = [c for c in df_subset.columns if any(c.startswith(o) for o in ohe_cols)]
for c in new_cols:
    print(f"   → {c}")

✅ Yes/No columns recoded to 1/0

Target distribution:
Ever_Depressive_Disorder
0.0    350778
1.0     85398
NaN      2517
Name: count, dtype: int64

 One-hot encoding done
   Shape after encoding: 438,693 rows x 97 columns

   New columns added:
   → Marital_Status_1.0
   → Marital_Status_2.0
   → Marital_Status_3.0
   → Marital_Status_4.0
   → Marital_Status_5.0
   → Marital_Status_6.0
   → Employment_Status_1.0
   → Employment_Status_2.0
   → Employment_Status_3.0
   → Employment_Status_4.0
   → Employment_Status_5.0
   → Employment_Status_6.0
   → Employment_Status_7.0
   → Employment_Status_8.0
   → Race_Ethnicity_1.0
   → Race_Ethnicity_2.0
   → Race_Ethnicity_3.0
   → Race_Ethnicity_4.0
   → Race_Ethnicity_5.0
   → Race_Ethnicity_9.0
   → Diabetes_Status_1.0
   → Diabetes_Status_2.0
   → Diabetes_Status_3.0
   → Diabetes_Status_4.0
   → State_1.0
   → State_2.0
   → State_4.0
   → State_5.0
   → State_6.0
   → State_8.0
   → State_9.0
   → State_10.0
   → State_11.0
   → State_13.

## Full Cleaning Pipeline (Consolidated)

The steps developed above are consolidated into a single reproducible pipeline cell. Running this cell alone reproduces the full cleaned dataset from scratch, making the notebook re-runnable without executing each exploration step. The pipeline runs 7 sequential steps: extract → filter target → replace codes → convert diet columns → recode binary columns → drop missing rows → one-hot encode.

**Row retention**: We start with 438,693 rows and retain 262,716 (60%) after dropping rows with any missing value. The 40% loss is largely driven by `Income_Level` (21.5% missing) and `BMI_Category` (10.7% missing) after the high-missing columns were already removed.

In [95]:
# ============================================================
# FULL CLEAN PIPELINE — Run this single cell
# This replaces all previous cleaning cells
# ============================================================

# Step 1 — Extract 25 columns from original df
cols_to_keep = {k: v for k, v in FINAL_VARIABLES.items() 
                if v not in ["Smoke_Frequency", "Binge_Drinking_Days", "Avg_Drinks_Per_Day"]}

df_clean = df[list(cols_to_keep.keys())].copy()
df_clean.rename(columns=cols_to_keep, inplace=True)
print(f"1. Extracted subset     : {df_clean.shape}")

# Step 2 — Filter target FIRST (before anything else)
df_clean = df_clean[df_clean["Ever_Depressive_Disorder"].isin([1.0, 2.0])].copy()
print(f"2. After target filter  : {df_clean.shape}")

# Step 3 — Replace special codes with NaN
binary_cols = [
    "Exercise_Past30Days", "Smoked_100_Cigarettes",
    "Ever_Heart_Disease", "Ever_Stroke", "Ever_COPD",
    "Ever_Arthritis", "Ever_Kidney_Disease",
    "Could_Not_Afford_Doctor", "Have_Health_Plan",
]
for col in binary_cols:
    df_clean[col] = df_clean[col].replace({7: np.nan, 9: np.nan})

df_clean["Diabetes_Status"]   = df_clean["Diabetes_Status"].replace({7: np.nan, 9: np.nan})
df_clean["General_Health"]    = df_clean["General_Health"].replace({7: np.nan, 9: np.nan})
df_clean["Income_Level"]      = df_clean["Income_Level"].replace({77: np.nan, 99: np.nan})
df_clean["Education_Level"]   = df_clean["Education_Level"].replace({9: np.nan})
df_clean["Marital_Status"]    = df_clean["Marital_Status"].replace({9: np.nan})
df_clean["Employment_Status"] = df_clean["Employment_Status"].replace({9: np.nan})
df_clean["Age_Group"]         = df_clean["Age_Group"].replace({14: np.nan})
df_clean["BMI_Category"]      = df_clean["BMI_Category"].replace({0: np.nan})

for col in ["Poor_Physical_Health_Days", "Poor_Mental_Health_Days"]:
    df_clean[col] = df_clean[col].replace({88: 0, 77: np.nan, 99: np.nan})

print(f"3. Special codes replaced")

# Step 4 — Convert Fruit & Vegetables
def convert_frequency(val):
    if pd.isna(val): return np.nan
    val = int(val)
    if val in [777, 999]:    return np.nan
    elif 100 <= val <= 199:  return val - 100
    elif 200 <= val <= 299:  return round((val - 200) / 7, 2)
    elif 300 <= val <= 399:  return round((val - 300) / 30, 2)
    else:                    return np.nan

df_clean["Fruit_Per_Day"]      = df_clean["Fruit_Per_Day"].apply(convert_frequency)
df_clean["Vegetables_Per_Day"] = df_clean["Vegetables_Per_Day"].apply(convert_frequency)
print(f"4. Fruit/Veg converted")

# Step 5 — Recode Yes/No columns (1→1, 2→0)
yes_no_cols = [
    "Exercise_Past30Days", "Smoked_100_Cigarettes",
    "Have_Health_Plan", "Could_Not_Afford_Doctor",
    "Ever_Heart_Disease", "Ever_Stroke", "Ever_COPD",
    "Ever_Arthritis", "Ever_Kidney_Disease",
]
for col in yes_no_cols:
    df_clean[col] = df_clean[col].map({1.0: 1, 2.0: 0})

# Recode target: 1=Depression, 0=No Depression
df_clean["Ever_Depressive_Disorder"] = df_clean["Ever_Depressive_Disorder"].map({1.0: 1, 2.0: 0})

# Recode Sex: 1=Male, 0=Female
df_clean["Sex"] = df_clean["Sex"].map({1.0: 1, 2.0: 0})

print(f"5. Yes/No columns recoded")

# Step 6 — Drop rows with missing values
before   = len(df_clean)
df_clean = df_clean.dropna().copy()
after    = len(df_clean)
print(f"6. After dropna         : {df_clean.shape} (dropped {before-after:,} rows)")

# Step 7 — One-hot encode categorical columns
ohe_cols = ["Marital_Status", "Employment_Status", "Race_Ethnicity", "Diabetes_Status", "State"]
df_clean = pd.get_dummies(df_clean, columns=ohe_cols, drop_first=False)
print(f"7. After encoding       : {df_clean.shape}")

# Final check
print(f"\n{'='*50}")
print(f"FINAL CLEAN DATASET")
print(f"{'='*50}")
print(f"Rows    : {df_clean.shape[0]:,}")
print(f"Columns : {df_clean.shape[1]}")
print(f"\nClass balance:")
print(df_clean["Ever_Depressive_Disorder"].value_counts())
print(df_clean["Ever_Depressive_Disorder"].value_counts(normalize=True).mul(100).round(1))

1. Extracted subset     : (438693, 25)
2. After target filter  : (436176, 25)
3. Special codes replaced
4. Fruit/Veg converted
5. Yes/No columns recoded
6. After dropna         : (262716, 25) (dropped 173,460 rows)
7. After encoding       : (262716, 97)

FINAL CLEAN DATASET
Rows    : 262,716
Columns : 97

Class balance:
Ever_Depressive_Disorder
0    209679
1     53037
Name: count, dtype: int64
Ever_Depressive_Disorder
0    79.8
1    20.2
Name: proportion, dtype: float64


## Save Clean Dataset

The fully cleaned and encoded dataset is saved as `BRFSS2021_clean.csv` before splitting. Saving at this point means train/test splits can be regenerated from this file without re-running the entire cleaning pipeline, and the clean dataset is available for exploratory analysis in separate notebooks.

In [96]:
# ============================================================
# SAVE — BRFSS2021_clean.csv (full cleaned dataset)
# Save BEFORE splitting
# ============================================================

CLEAN_CSV = PROCESSED_DIR / "BRFSS2021_clean.csv"
df_clean.to_csv(CLEAN_CSV, index=False)

print(f"✅ Clean dataset saved")
print(f"   Rows    : {df_clean.shape[0]:,}")
print(f"   Columns : {df_clean.shape[1]}")
print(f"   Location: {CLEAN_CSV}")

✅ Clean dataset saved
   Rows    : 262,716
   Columns : 97
   Location: /Users/ishamandawkar/PulsePredictor/data/processed/BRFSS2021_clean.csv


## Train / Test Split (80/20, Stratified)

The clean dataset is split into training (80%) and test (20%) sets using stratification on the target variable. Stratification ensures both sets maintain the same class ratio (~79.8% No Depression / 20.2% Depression), preventing a situation where one set is accidentally more imbalanced than the other. `random_state=42` ensures reproducibility. All four splits are saved to disk for use in subsequent modeling notebooks.

In [97]:
# ============================================================
# MILESTONE 5 — Train / Test Split (80/20)
# ============================================================
from sklearn.model_selection import train_test_split

# Separate features and target
X = df_clean.drop(columns=["Ever_Depressive_Disorder"])
y = df_clean["Ever_Depressive_Disorder"]

# stratify=y ensures 80/20 class balance is preserved in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"✅ Train set : {X_train.shape[0]:,} rows x {X_train.shape[1]} columns")
print(f"✅ Test set  : {X_test.shape[0]:,} rows x {X_test.shape[1]} columns")

print(f"\nClass balance in train set:")
print(y_train.value_counts(normalize=True).mul(100).round(1))

print(f"\nClass balance in test set:")
print(y_test.value_counts(normalize=True).mul(100).round(1))

# Save all splits
X_train.to_csv(PROCESSED_DIR / "X_train.csv", index=False)
X_test.to_csv(PROCESSED_DIR / "X_test.csv", index=False)
y_train.to_csv(PROCESSED_DIR / "y_train.csv", index=False)
y_test.to_csv(PROCESSED_DIR / "y_test.csv", index=False)

print(f"\n✅ All splits saved to: {PROCESSED_DIR}")

✅ Train set : 210,172 rows x 96 columns
✅ Test set  : 52,544 rows x 96 columns

Class balance in train set:
Ever_Depressive_Disorder
0    79.8
1    20.2
Name: proportion, dtype: float64

Class balance in test set:
Ever_Depressive_Disorder
0    79.8
1    20.2
Name: proportion, dtype: float64

✅ All splits saved to: /Users/ishamandawkar/PulsePredictor/data/processed


# Data Cleaning Documentation
## BRFSS 2021 — Depression Risk Factor Analysis

### Dataset Source
- **Raw data**: LLCP2021.XPT from CDC BRFSS 2021 survey
- **Original size**: 438,693 rows × 303 columns
- **Final clean size**: 262,716 rows × 97 columns

### Variable Selection
28 variables selected covering:
- Target: Ever diagnosed with depressive disorder (ADDEPEV3)
- Demographics: Age, Sex, Education, Income, Marital Status, Employment, Race, State
- Lifestyle: Exercise, Smoking, Diet (Fruit & Vegetables)
- Health Status: BMI, General Health, Physical/Mental Health Days, Health Plan, Doctor Affordability
- Chronic Conditions: Heart Disease, Stroke, Diabetes, COPD, Arthritis, Kidney Disease

3 variables dropped due to >50% missing values:
- Smoke_Frequency (61.9% missing)
- Binge_Drinking_Days (52.3% missing)
- Avg_Drinks_Per_Day (52.2% missing)

### Cleaning Steps
1. Filtered target variable to valid responses only (Yes/No), removed Don't Know and Refused
2. Replaced BRFSS special codes with NaN (7/9 = Don't Know/Refused, 77/99 = same for 2-digit, 88 = None recoded to 0)
3. Converted Fruit and Vegetables columns from BRFSS frequency codes to daily estimates
4. Dropped 173,460 rows with any remaining missing values
5. Recoded all Yes/No columns from (1/2) to (1/0)
6. One-hot encoded 5 categorical columns: Marital Status, Employment Status, Race/Ethnicity, Diabetes Status, State

### Encoding Details
- **Binary recoded (1=Yes, 0=No)**: Exercise, Smoking, Health Plan, Doctor Affordability, all chronic conditions
- **Ordinal kept as-is**: Age Group, Education, Income, General Health, BMI Category
- **One-hot encoded**: Marital Status (6), Employment Status (8), Race/Ethnicity (6), Diabetes Status (4), State (53)

### Class Balance
| Class | Count | Percentage |
|-------|-------|------------|
| No Depression (0) | 209,679 | 79.8% |
| Depression (1) | 53,037 | 20.2% |

Note: Dataset is moderately imbalanced. Will handle using class_weight='balanced' during modeling.

### Train/Test Split
- **Method**: Stratified 80/20 split (random_state=42)
- **Train set**: 210,172 rows
- **Test set**: 52,544 rows
- **Stratified**: Yes — class balance preserved in both sets

### Output Files
| File | Description |
|------|-------------|
| BRFSS2021_clean.csv | Full cleaned and encoded dataset |
| X_train.csv | Training features |
| X_test.csv | Test features |
| y_train.csv | Training labels |
| y_test.csv | Test labels |